# 🏀 NBA Boxscore Deep EDA & Data Cleaning

This notebook provides comprehensive exploratory data analysis and cleaning of the NBA boxscore raw dataset. It systematically examines data quality, identifies issues, and implements cleaning strategies to prepare the data for analytics.

## Table of Contents
1. **Import Libraries & Load Data**
2. **Boxscore JSON Parser Functions**
3. **Basic Data Overview & Structure**
4. **Column-by-Column Deep Dive**
5. **Missing Values Analysis**
6. **Data Type Validation**
7. **Duplicate Records Detection**
8. **Outlier Detection & Analysis**
9. **Data Quality Issues**
10. **Text Data Cleaning**
11. **Categorical Data Analysis**
12. **Numerical Data Distribution**
13. **Cross-Column Validation**
14. **Data Cleaning Implementation**
15. **Cleaned Data Validation & Export**

---

**Dataset**: 2024 NBA Boxscores  
**Objective**: Clean and prepare boxscore data for Pacers analytics pipeline  
**Tech Stack**: pandas, numpy, matplotlib, seaborn, scipy, json

## 1. Import Required Libraries and Load Data

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import os
import json
from pathlib import Path

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Statistical analysis
from scipy import stats
from scipy.stats import zscore

# Data quality and missing data analysis
import missingno as msno

## 2. Boxscore JSON Parser Functions

Comprehensive functions to parse complex boxscore JSON data into readable DataFrames.

In [ ]:
def parse_boxscore_data(parquet_file):
    """
    Parse complex boxscore parquet data into readable DataFrames.
    
    Parameters:
    parquet_file (str): Path to the boxscore parquet file
    
    Returns:
    dict: Dictionary containing multiple DataFrames:
        - 'game_info': Basic game information
        - 'team_stats': Team-level statistics
        - 'player_stats': Individual player statistics
        - 'raw_data': Original data for reference
    """
    
    # Read the parquet file
    df = pd.read_parquet(parquet_file)
    
    result = {}
    all_game_info = []
    all_team_stats = []
    all_player_stats = []
    
    print(f"📊 Processing {len(df)} games from {Path(parquet_file).name}")
    
    for idx, row in df.iterrows():
        try:
            game_id = row.get('gameId')
            home_team_id = row.get('homeTeam_teamId')
            away_team_id = row.get('awayTeam_teamId')
            
            # Extract basic game information
            game_info = {
                'game_id': game_id,
                'home_team_id': home_team_id,
                'away_team_id': away_team_id,
                'file_source': Path(parquet_file).name
            }
            all_game_info.append(game_info)
            
            # Extract team statistics
            home_stats = {'game_id': game_id, 'team_type': 'home', 'team_id': home_team_id}
            away_stats = {'game_id': game_id, 'team_type': 'away', 'team_id': away_team_id}
            
            # Add all team statistics
            for key, value in row.items():
                if key.startswith('homeTeam_statistics_'):
                    stat_name = key.replace('homeTeam_statistics_', '')
                    home_stats[stat_name] = value
                elif key.startswith('awayTeam_statistics_'):
                    stat_name = key.replace('awayTeam_statistics_', '')
                    away_stats[stat_name] = value
            
            all_team_stats.extend([home_stats, away_stats])
            
            # Extract player statistics - FIXED PARSING
            # Home team players
            if 'homeTeam_players' in row and row['homeTeam_players'] is not None:
                home_players_array = row['homeTeam_players']
                
                # Convert numpy array to list if needed
                if isinstance(home_players_array, np.ndarray):
                    home_players_list = home_players_array.tolist()
                else:
                    home_players_list = home_players_array if home_players_array else []
                
                # Process each player
                for player in home_players_list:
                    if isinstance(player, dict):
                        player_stats = {
                            'game_id': game_id,
                            'team_type': 'home',
                            'team_id': home_team_id,
                            'player_id': player.get('personId'),
                            'player_name': player.get('name'),
                            'first_name': player.get('firstName'),
                            'family_name': player.get('familyName'),
                            'name_i': player.get('nameI'),
                            'jersey_num': player.get('jerseyNum'),
                            'position': player.get('position'),
                            'starter': player.get('starter'),
                            'played': player.get('played'),
                            'status': player.get('status'),
                            'oncourt': player.get('oncourt'),
                            'not_playing_reason': player.get('notPlayingReason'),
                            'not_playing_description': player.get('notPlayingDescription'),
                            'order': player.get('order')
                        }
                        
                        # Add player statistics
                        if 'statistics' in player and isinstance(player['statistics'], dict):
                            for stat_name, stat_value in player['statistics'].items():
                                player_stats[stat_name] = stat_value
                        
                        all_player_stats.append(player_stats)
            
            # Away team players
            if 'awayTeam_players' in row and row['awayTeam_players'] is not None:
                away_players_array = row['awayTeam_players']
                
                # Convert numpy array to list if needed
                if isinstance(away_players_array, np.ndarray):
                    away_players_list = away_players_array.tolist()
                else:
                    away_players_list = away_players_array if away_players_array else []
                
                # Process each player
                for player in away_players_list:
                    if isinstance(player, dict):
                        player_stats = {
                            'game_id': game_id,
                            'team_type': 'away',
                            'team_id': away_team_id,
                            'player_id': player.get('personId'),
                            'player_name': player.get('name'),
                            'first_name': player.get('firstName'),
                            'family_name': player.get('familyName'),
                            'name_i': player.get('nameI'),
                            'jersey_num': player.get('jerseyNum'),
                            'position': player.get('position'),
                            'starter': player.get('starter'),
                            'played': player.get('played'),
                            'status': player.get('status'),
                            'oncourt': player.get('oncourt'),
                            'not_playing_reason': player.get('notPlayingReason'),
                            'not_playing_description': player.get('notPlayingDescription'),
                            'order': player.get('order')
                        }
                        
                        # Add player statistics
                        if 'statistics' in player and isinstance(player['statistics'], dict):
                            for stat_name, stat_value in player['statistics'].items():
                                player_stats[stat_name] = stat_value
                        
                        all_player_stats.append(player_stats)
                    
        except Exception as e:
            print(f"⚠️  Error processing row {idx}: {e}")
            continue
    
    # Create DataFrames
    result['game_info'] = pd.DataFrame(all_game_info)
    result['team_stats'] = pd.DataFrame(all_team_stats)
    result['player_stats'] = pd.DataFrame(all_player_stats)
    result['raw_data'] = df
    
    # Print summary
    print(f"✅ Parsing complete!")
    print(f"   📋 Games processed: {len(result['game_info'])}")
    print(f"   🏀 Team stat records: {len(result['team_stats'])}")
    print(f"   👤 Player stat records: {len(result['player_stats'])}")
    
    return result

def load_and_parse_boxscore_files(boxscore_directory):
    """
    Load and parse all boxscore parquet files from a directory.
    
    Parameters:
    boxscore_directory (str): Path to directory containing boxscore parquet files
    
    Returns:
    dict: Combined DataFrames from all files
    """
    
    import glob
    
    boxscore_files = glob.glob(f"{boxscore_directory}/*.parquet")
    
    if not boxscore_files:
        print(f"❌ No parquet files found in {boxscore_directory}")
        return None
    
    print(f"🔍 Found {len(boxscore_files)} boxscore files")
    
    all_game_info = []
    all_team_stats = []
    all_player_stats = []
    
    for i, file_path in enumerate(boxscore_files[:5], 1):  # Process first 5 files for demonstration
        print(f"\n📁 Processing file {i}/{min(5, len(boxscore_files))}: {Path(file_path).name}")
        
        try:
            file_data = parse_boxscore_data(file_path)
            
            all_game_info.append(file_data['game_info'])
            all_team_stats.append(file_data['team_stats'])
            all_player_stats.append(file_data['player_stats'])
            
        except Exception as e:
            print(f"⚠️  Error processing {file_path}: {e}")
            continue
    
    # Combine all DataFrames
    combined_data = {
        'game_info': pd.concat(all_game_info, ignore_index=True) if all_game_info else pd.DataFrame(),
        'team_stats': pd.concat(all_team_stats, ignore_index=True) if all_team_stats else pd.DataFrame(),
        'player_stats': pd.concat(all_player_stats, ignore_index=True) if all_player_stats else pd.DataFrame()
    }
    
    print(f"\n🎯 COMBINED RESULTS:")
    print(f"   📋 Total games: {len(combined_data['game_info'])}")
    print(f"   🏀 Total team records: {len(combined_data['team_stats'])}")
    print(f"   👤 Total player records: {len(combined_data['player_stats'])}")
    
    return combined_data

def display_boxscore_summary(parsed_data):
    """
    Display a comprehensive summary of the parsed boxscore data.
    
    Parameters:
    parsed_data (dict): Dictionary containing the parsed DataFrames
    """
    
    print("📊 BOXSCORE DATA SUMMARY")
    print("=" * 40)
    
    for df_name, df in parsed_data.items():
        if df_name != 'raw_data' and isinstance(df, pd.DataFrame) and len(df) > 0:
            print(f"\n🔍 {df_name.upper()} DATA:")
            print(f"   Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
            print(f"   Columns: {', '.join(df.columns[:5])}{'...' if len(df.columns) > 5 else ''}")
            
            # Show sample data
            print(f"   Sample data:")
            display(df.head(2))
    
    # Show column details for each DataFrame
    print(f"\n📋 DETAILED COLUMN INFORMATION:")
    print("-" * 50)
    
    for df_name, df in parsed_data.items():
        if df_name != 'raw_data' and isinstance(df, pd.DataFrame) and len(df) > 0:
            print(f"\n{df_name.upper()} COLUMNS ({len(df.columns)} total):")
            for i, col in enumerate(df.columns, 1):
                dtype = df[col].dtype
                null_count = df[col].isnull().sum()
                unique_count = df[col].nunique()
                print(f"  {i:2d}. {col:<25} | {str(dtype):<10} | Nulls: {null_count:4d} | Unique: {unique_count:4d}")

print("✅ FULLY FIXED Boxscore parsing functions loaded!")
print("📝 Available functions:")
print("   • parse_boxscore_data(parquet_file) - Parse single file")
print("   • load_and_parse_boxscore_files(directory) - Parse all files in directory")
print("   • display_boxscore_summary(parsed_data) - Show comprehensive summary")
print("\n🔧 FINAL FIXES APPLIED:")
print("   • Removed problematic team_name references that don't exist in data")
print("   • Simplified variable handling to avoid naming conflicts")
print("   • Proper handling of numpy arrays in player data")
print("   • Correct extraction of player statistics from nested structure")
print("   • Added all player info fields (nameI, oncourt, etc.)")

In [ ]:
# Navigate to project root if needed
current_dir = Path.cwd()
if current_dir.name == 'notebooks':
    os.chdir(current_dir.parent)
    print(f"📁 Changed directory to: {Path.cwd()}")

# Load and parse boxscore data
boxscore_directory = 'data/raw/Boxscores'

if Path(boxscore_directory).exists():
    print("🏀 Loading and parsing NBA boxscore data...")
    
    # Parse boxscore files
    boxscore_data = load_and_parse_boxscore_files(boxscore_directory)
    
    if boxscore_data:
        # Display comprehensive summary
        display_boxscore_summary(boxscore_data)
        
        # Store individual DataFrames for easy access
        games_df = boxscore_data['game_info']
        team_stats_df = boxscore_data['team_stats']
        player_stats_df = boxscore_data['player_stats']
        
        print(f"\n💾 DataFrames created:")
        print(f"   📋 games_df: Game information ({len(games_df)} games)")
        print(f"   🏀 team_stats_df: Team statistics ({len(team_stats_df)} team records)")
        print(f"   👤 player_stats_df: Player statistics ({len(player_stats_df)} player records)")
        
        # Create backup copies for cleaning comparison
        games_original = games_df.copy()
        team_stats_original = team_stats_df.copy()
        player_stats_original = player_stats_df.copy()
        
        print(f"\n💾 Created backup copies for comparison")
        
    else:
        print("❌ Failed to parse boxscore data")
        
else:
    print(f"❌ Boxscore directory not found: {boxscore_directory}")
    print("Please ensure the boxscore data exists in the correct location.")

In [ ]:
# Quick Analysis Functions for Boxscore Data
def get_pacers_data():
    """
    Extract Indiana Pacers specific data from the datasets.
    
    Returns:
    dict: Dictionary containing Pacers-specific DataFrames
    """
    
    # Check if data is loaded
    if not all(var in globals() for var in ['games_df', 'team_stats_df', 'player_stats_df']):
        print("❌ Boxscore data not loaded. Please run the data loading cell first.")
        return None
    
    # Pacers team ID
    pacers_team_id = 1610612754  # Indiana Pacers team ID
    
    # Get Pacers players
    pacers_players = player_stats_df[player_stats_df['team_id'] == pacers_team_id].copy()
    
    # Get Pacers team stats
    pacers_team_stats = team_stats_df[team_stats_df['team_id'] == pacers_team_id].copy()
    
    # Get games where Pacers played
    pacers_games = games_df[
        (games_df['home_team_id'] == pacers_team_id) | 
        (games_df['away_team_id'] == pacers_team_id)
    ].copy()
    
    result = {
        'players': pacers_players,
        'team_stats': pacers_team_stats,
        'games': pacers_games
    }
    
    print(f"🏀 PACERS DATA EXTRACTED:")
    print(f"   👤 Player records: {len(pacers_players)}")
    print(f"   🏀 Team stat records: {len(pacers_team_stats)}")
    print(f"   📅 Games: {len(pacers_games)}")
    
    return result

def analyze_player_performance(min_minutes=10):
    """
    Analyze player performance from the boxscore data.
    
    Parameters:
    min_minutes (int): Minimum minutes played filter
    
    Returns:
    DataFrame: Summary of player performance
    """
    
    # Check if data is loaded
    if 'player_stats_df' not in globals():
        print("❌ Player stats data not loaded. Please run the data loading cell first.")
        return pd.DataFrame()
    
    # Filter players who played significant minutes
    active_players = player_stats_df[
        (player_stats_df['played'] == '1') & 
        (player_stats_df['points'].notna())
    ].copy()
    
    if len(active_players) == 0:
        print("No active player data found")
        return pd.DataFrame()
    
    # Create performance summary
    performance_summary = active_players.groupby(['player_name', 'team_id']).agg({
        'points': ['mean', 'max', 'sum', 'count'],
        'assists': ['mean', 'max', 'sum'],
        'reboundsTotal': ['mean', 'max', 'sum'],
        'steals': ['mean', 'sum'],
        'blocks': ['mean', 'sum'],
        'turnovers': ['mean', 'sum'],
        'fieldGoalsPercentage': 'mean',
        'threePointersPercentage': 'mean',
        'freeThrowsPercentage': 'mean'
    }).round(2)
    
    # Flatten column names
    performance_summary.columns = ['_'.join(col).strip() for col in performance_summary.columns]
    performance_summary = performance_summary.reset_index()
    
    print(f"📊 PLAYER PERFORMANCE ANALYSIS:")
    print(f"   👤 Players analyzed: {len(performance_summary)}")
    print(f"   🎯 Total player records: {len(active_players)}")
    
    return performance_summary

def get_column_info_all_tables():
    """
    Get comprehensive column information for all parsed tables.
    """
    
    # Check if data is loaded
    if not all(var in globals() for var in ['games_df', 'team_stats_df', 'player_stats_df']):
        print("❌ Boxscore data not loaded. Please run the data loading cell first.")
        return
        
    tables = {
        'games_df': games_df,
        'team_stats_df': team_stats_df, 
        'player_stats_df': player_stats_df
    }
    
    print("📋 COLUMN INFORMATION FOR ALL TABLES")
    print("=" * 50)
    
    for table_name, df in tables.items():
        if len(df) > 0:
            print(f"\n🔍 {table_name.upper()} ({df.shape[0]:,} rows × {df.shape[1]} columns)")
            print("-" * 40)
            
            for i, col in enumerate(df.columns, 1):
                dtype = df[col].dtype
                null_count = df[col].isnull().sum()
                null_pct = (null_count / len(df)) * 100
                unique_count = df[col].nunique()
                
                # Sample values
                sample_vals = df[col].dropna().head(2).tolist()
                sample_str = str(sample_vals)[:30] + "..." if len(str(sample_vals)) > 30 else str(sample_vals)
                
                print(f"{i:2d}. {col:<25} | {str(dtype):<12} | Nulls: {null_count:4d} ({null_pct:5.1f}%) | Unique: {unique_count:4d} | Sample: {sample_str}")

def quick_data_check():
    """
    Quick check of the loaded data status and basic info.
    """
    print("🔍 QUICK DATA STATUS CHECK")
    print("=" * 30)
    
    data_vars = ['games_df', 'team_stats_df', 'player_stats_df']
    
    for var_name in data_vars:
        if var_name in globals():
            df = globals()[var_name]
            print(f"✅ {var_name}: {df.shape[0]:,} rows × {df.shape[1]} columns")
        else:
            print(f"❌ {var_name}: Not loaded")
    
    # Quick sample if player data exists
    if 'player_stats_df' in globals() and len(player_stats_df) > 0:
        active_count = len(player_stats_df[player_stats_df['played'] == '1'])
        print(f"\n👤 Active players (played='1'): {active_count}")
        
        # Check for Pacers
        pacers_count = len(player_stats_df[player_stats_df['team_id'] == 1610612754])
        print(f"🏀 Pacers players: {pacers_count}")

print("✅ UPDATED Analysis functions loaded!")
print("📝 Available analysis functions:")
print("   • get_pacers_data() - Extract Pacers-specific data")
print("   • analyze_player_performance() - Player performance analysis")
print("   • get_column_info_all_tables() - Detailed column information")
print("   • quick_data_check() - Check data loading status")

In [ ]:
# Load and Parse Boxscore Data
print("🔄 Loading boxscore data...")

try:
    # Load boxscore data
    boxscore_data = load_and_parse_boxscore_files()
    
    if boxscore_data and len(boxscore_data) > 0:
        # Parse all data
        all_parsed = []
        
        print(f"📊 Processing {len(boxscore_data)} boxscore files...")
        
        for filename, data in boxscore_data.items():
            try:
                parsed = parse_boxscore_data(data, filename)
                if parsed:
                    all_parsed.append(parsed)
            except Exception as e:
                print(f"❌ Error parsing {filename}: {e}")
        
        if all_parsed:
            # Combine all parsed data
            games_df = pd.concat([p['games'] for p in all_parsed], ignore_index=True)
            team_stats_df = pd.concat([p['team_stats'] for p in all_parsed], ignore_index=True)
            player_stats_df = pd.concat([p['player_stats'] for p in all_parsed], ignore_index=True)
            
            print("\n✅ BOXSCORE DATA SUCCESSFULLY LOADED!")
            print("=" * 50)
            print(f"🎮 Games DataFrame: {games_df.shape[0]:,} rows × {games_df.shape[1]} columns")
            print(f"🏀 Team Stats DataFrame: {team_stats_df.shape[0]:,} rows × {team_stats_df.shape[1]} columns")
            print(f"👤 Player Stats DataFrame: {player_stats_df.shape[0]:,} rows × {player_stats_df.shape[1]} columns")
            
            # Display summary
            display_boxscore_summary(games_df, team_stats_df, player_stats_df)
            
        else:
            print("❌ No data could be parsed from boxscore files")
    else:
        print("❌ No boxscore data found to load")
        
except Exception as e:
    print(f"❌ Error loading boxscore data: {e}")
    print("Traceback:")
    import traceback
    traceback.print_exc()

In [ ]:
# Let's examine the actual structure of the boxscore data
print("🔍 Examining boxscore data structure...")

# Load a sample file to understand the structure
sample_file = 'data/raw/Boxscores/2025_0042400101_NBA_Traditional_boxscores.parquet'

if Path(sample_file).exists():
    # Read the sample file
    sample_df = pd.read_parquet(sample_file)
    
    print(f"📊 Sample file shape: {sample_df.shape}")
    print(f"📋 Columns ({len(sample_df.columns)}):")
    
    # Show all columns
    for i, col in enumerate(sample_df.columns, 1):
        print(f"  {i:2d}. {col}")
    
    print(f"\n🎯 Looking for player-related columns:")
    player_cols = [col for col in sample_df.columns if 'player' in col.lower()]
    for col in player_cols:
        print(f"   • {col}")
        
    print(f"\n🏀 Looking for team-related columns:")
    team_cols = [col for col in sample_df.columns if 'team' in col.lower()]
    for col in team_cols[:10]:  # Show first 10
        print(f"   • {col}")
    
    # Show sample data structure
    print(f"\n📋 Sample row data (first row):")
    first_row = sample_df.iloc[0]
    
    # Look for player data specifically
    for col in sample_df.columns:
        if 'player' in col.lower():
            print(f"\n🔍 {col}:")
            print(f"   Type: {type(first_row[col])}")
            print(f"   Value: {str(first_row[col])[:200]}...")
            
    # Look for team player arrays
    for col in sample_df.columns:
        if 'Team_players' in col:
            print(f"\n🏀 {col}:")
            print(f"   Type: {type(first_row[col])}")
            if isinstance(first_row[col], list) and len(first_row[col]) > 0:
                print(f"   Length: {len(first_row[col])}")
                print(f"   First player keys: {list(first_row[col][0].keys()) if isinstance(first_row[col][0], dict) else 'Not a dict'}")
                
else:
    print(f"❌ Sample file not found: {sample_file}")

In [ ]:
# Let's look specifically at player data structure
print("🔍 Focused examination of player data structure...")

if 'sample_df' in locals():
    first_row = sample_df.iloc[0]
    
    # Find the correct player columns
    player_related_cols = [col for col in sample_df.columns if 'players' in col or 'Player' in col]
    print(f"📋 Player-related columns found: {player_related_cols}")
    
    for col in player_related_cols:
        if first_row[col] is not None:
            print(f"\n🏀 {col}:")
            print(f"   Type: {type(first_row[col])}")
            
            if isinstance(first_row[col], list):
                print(f"   Length: {len(first_row[col])}")
                if len(first_row[col]) > 0:
                    print(f"   First player type: {type(first_row[col][0])}")
                    if isinstance(first_row[col][0], dict):
                        print(f"   First player keys: {list(first_row[col][0].keys())}")
                        # Show sample statistics structure
                        if 'statistics' in first_row[col][0]:
                            stats = first_row[col][0]['statistics']
                            print(f"   Statistics type: {type(stats)}")
                            if isinstance(stats, dict):
                                print(f"   Statistics keys: {list(stats.keys())[:10]}...")  # Show first 10 stats
            break
    
    # Also check team statistics structure
    team_stats_cols = [col for col in sample_df.columns if 'statistics' in col and 'Team' in col]
    print(f"\n🏀 Team statistics columns: {team_stats_cols[:5]}...")  # Show first 5
    
else:
    print("❌ No sample data loaded")

In [ ]:
# Examine the numpy array structure of player data
print("🔍 Examining numpy array structure...")

if 'sample_df' in locals():
    first_row = sample_df.iloc[0]
    
    # Look at homeTeam_players numpy array
    home_players = first_row['homeTeam_players']
    print(f"🏠 Home team players:")
    print(f"   Type: {type(home_players)}")
    print(f"   Shape: {home_players.shape if hasattr(home_players, 'shape') else 'No shape'}")
    print(f"   Length: {len(home_players) if hasattr(home_players, '__len__') else 'No length'}")
    
    # Convert to list and examine
    if isinstance(home_players, np.ndarray):
        home_players_list = home_players.tolist()
        print(f"   Converted to list length: {len(home_players_list)}")
        
        if len(home_players_list) > 0:
            first_player = home_players_list[0]
            print(f"   First player type: {type(first_player)}")
            
            if isinstance(first_player, dict):
                print(f"   First player keys: {list(first_player.keys())}")
                
                # Look at statistics
                if 'statistics' in first_player:
                    stats = first_player['statistics']
                    print(f"   Statistics type: {type(stats)}")
                    if isinstance(stats, dict):
                        print(f"   Statistics keys ({len(stats)}): {list(stats.keys())}")
                        
                        # Show first few stat values
                        stat_sample = dict(list(stats.items())[:5])
                        print(f"   Sample stats: {stat_sample}")
                        
                # Show player info
                player_info = {k: v for k, v in first_player.items() if k != 'statistics'}
                print(f"   Player info: {player_info}")
    
    print(f"\n🔄 Same for away team...")
    away_players = first_row['awayTeam_players']
    if isinstance(away_players, np.ndarray):
        away_players_list = away_players.tolist()
        print(f"   Away team players count: {len(away_players_list)}")
        
else:
    print("❌ No sample data loaded")

In [ ]:
# Verify the player stats parsing is working correctly
print("✅ VERIFYING PLAYER STATS PARSING")
print("=" * 40)

if 'player_stats_df' in locals() and len(player_stats_df) > 0:
    
    print(f"📊 Player Stats DataFrame: {player_stats_df.shape[0]:,} rows × {player_stats_df.shape[1]} columns")
    
    # Show sample player data
    print(f"\n🏀 Sample player records:")
    sample_players = player_stats_df.head(3)[['player_name', 'team_name', 'position', 'points', 'assists', 'reboundsTotal', 'minutes']]
    display(sample_players)
    
    # Check for statistical columns
    stat_columns = [col for col in player_stats_df.columns if col in [
        'points', 'assists', 'reboundsTotal', 'steals', 'blocks', 'turnovers',
        'fieldGoalsMade', 'fieldGoalsAttempted', 'fieldGoalsPercentage',
        'threePointersMade', 'threePointersAttempted', 'threePointersPercentage',
        'freeThrowsMade', 'freeThrowsAttempted', 'freeThrowsPercentage', 'minutes'
    ]]
    
    print(f"\n📈 Statistical columns found ({len(stat_columns)}):")
    for col in stat_columns:
        print(f"   • {col}")
    
    # Check players who actually played
    active_players = player_stats_df[player_stats_df['played'] == '1']
    print(f"\n👤 Active players (played = '1'): {len(active_players)}")
    
    if len(active_players) > 0:
        # Show some basic stats
        print(f"\n📊 Basic statistics for active players:")
        print(f"   • Average points: {active_players['points'].mean():.1f}")
        print(f"   • Average assists: {active_players['assists'].mean():.1f}")
        print(f"   • Average rebounds: {active_players['reboundsTotal'].mean():.1f}")
        
        # Top scorers
        top_scorers = active_players.nlargest(5, 'points')[['player_name', 'team_name', 'points', 'assists', 'reboundsTotal']]
        print(f"\n🏆 Top 5 scorers:")
        display(top_scorers)
    
    # Check for Pacers players (team_id 1610612754)
    pacers_players = player_stats_df[player_stats_df['team_id'] == 1610612754]
    print(f"\n🏀 Indiana Pacers players found: {len(pacers_players)}")
    
    if len(pacers_players) > 0:
        pacers_active = pacers_players[pacers_players['played'] == '1']
        print(f"   • Active Pacers players: {len(pacers_active)}")
        if len(pacers_active) > 0:
            print(f"   • Pacers players sample:")
            pacers_sample = pacers_active[['player_name', 'position', 'points', 'assists', 'reboundsTotal']].head(3)
            display(pacers_sample)
    
else:
    print("❌ No player stats data available")

In [ ]:
# Quick data check and player stats verification
quick_data_check()

# If data is loaded, show some player stats
if 'player_stats_df' in globals() and len(player_stats_df) > 0:
    print(f"\n🏀 PLAYER STATS SAMPLE:")
    print("-" * 30)
    
    # Show first few players with their stats
    sample_cols = ['player_name', 'team_name', 'position', 'played', 'points', 'assists', 'reboundsTotal', 'minutes']
    available_cols = [col for col in sample_cols if col in player_stats_df.columns]
    
    if available_cols:
        sample_data = player_stats_df[available_cols].head(5)
        display(sample_data)
    
    # Show all statistical columns available
    stat_cols = [col for col in player_stats_df.columns if col in [
        'points', 'assists', 'reboundsTotal', 'steals', 'blocks', 'turnovers',
        'fieldGoalsMade', 'fieldGoalsAttempted', 'fieldGoalsPercentage',
        'threePointersMade', 'threePointersAttempted', 'threePointersPercentage',
        'freeThrowsMade', 'freeThrowsAttempted', 'freeThrowsPercentage', 'minutes',
        'reboundsDefensive', 'reboundsOffensive', 'foulsPersonal', 'plusMinusPoints'
    ]]
    
    print(f"\n📊 Available statistical columns ({len(stat_cols)}):")
    for col in stat_cols:
        print(f"   • {col}")
        
else:
    print("\n❌ No player stats data found. Checking what data is available...")
    
    # Check what variables exist in the global scope
    boxscore_vars = [var for var in globals().keys() if 'df' in var or 'data' in var]
    print(f"Available data variables: {boxscore_vars}")

In [ ]:
# Debug the parsing issue
print("🔍 DEBUGGING PLAYER STATS PARSING")
print("=" * 40)

# Check the boxscore_data that was loaded
if 'boxscore_data' in globals():
    print(f"📊 Boxscore data keys: {list(boxscore_data.keys())}")
    
    for key, df in boxscore_data.items():
        print(f"   • {key}: {df.shape}")
        
    # Check if player_stats has data
    if 'player_stats' in boxscore_data:
        player_df = boxscore_data['player_stats']
        print(f"\n👤 Player stats shape: {player_df.shape}")
        
        if len(player_df) > 0:
            print(f"   • Columns: {list(player_df.columns)[:10]}...")
            print(f"   • Sample data:")
            display(player_df.head(2))
        else:
            print("   • Player stats DataFrame is empty!")
            
            # Let's re-examine the raw data structure
            print(f"\n🔍 Re-examining raw data structure...")
            if 'sample_df' in globals():
                first_row = sample_df.iloc[0]
                
                # Check if players arrays have data
                home_players = first_row.get('homeTeam_players')
                away_players = first_row.get('awayTeam_players')
                
                print(f"   • Home players type: {type(home_players)}")
                print(f"   • Home players length: {len(home_players) if hasattr(home_players, '__len__') else 'No length'}")
                
                if isinstance(home_players, np.ndarray) and len(home_players) > 0:
                    home_list = home_players.tolist()
                    print(f"   • First home player: {home_list[0] if len(home_list) > 0 else 'None'}")
                
                print(f"   • Away players type: {type(away_players)}")
                print(f"   • Away players length: {len(away_players) if hasattr(away_players, '__len__') else 'No length'}")
else:
    print("❌ No boxscore_data found")

In [ ]:
# Let's test the parsing function on a single file with debug output
print("🔧 TESTING PARSING FUNCTION WITH DEBUG")
print("=" * 40)

def debug_parse_single_file(file_path):
    """Debug version of parse function with verbose output"""
    
    df = pd.read_parquet(file_path)
    print(f"📁 Loaded file: {Path(file_path).name}")
    print(f"   Shape: {df.shape}")
    
    all_player_stats = []
    
    for idx, row in df.iterrows():
        print(f"\n🔍 Processing row {idx}:")
        
        # Check home team players
        if 'homeTeam_players' in row:
            home_players = row['homeTeam_players']
            print(f"   🏠 Home players type: {type(home_players)}")
            print(f"   🏠 Home players length: {len(home_players) if hasattr(home_players, '__len__') else 'No length'}")
            
            if home_players is not None:
                if isinstance(home_players, np.ndarray):
                    home_players_list = home_players.tolist()
                    print(f"   🏠 Converted to list: {len(home_players_list)} players")
                    
                    for i, player in enumerate(home_players_list):
                        if isinstance(player, dict):
                            print(f"      Player {i+1}: {player.get('name', 'Unknown')}")
                            all_player_stats.append(player)  # Simplified for debug
                        else:
                            print(f"      Player {i+1}: Not a dict - {type(player)}")
                else:
                    print(f"   🏠 Not a numpy array: {type(home_players)}")
            else:
                print(f"   🏠 Home players is None")
        else:
            print(f"   🏠 No 'homeTeam_players' column found")
            
        # Check away team players
        if 'awayTeam_players' in row:
            away_players = row['awayTeam_players']
            print(f"   ✈️  Away players type: {type(away_players)}")
            print(f"   ✈️  Away players length: {len(away_players) if hasattr(away_players, '__len__') else 'No length'}")
            
            if away_players is not None:
                if isinstance(away_players, np.ndarray):
                    away_players_list = away_players.tolist()
                    print(f"   ✈️  Converted to list: {len(away_players_list)} players")
                    
                    for i, player in enumerate(away_players_list):
                        if isinstance(player, dict):
                            print(f"      Player {i+1}: {player.get('name', 'Unknown')}")
                            all_player_stats.append(player)  # Simplified for debug
                        else:
                            print(f"      Player {i+1}: Not a dict - {type(player)}")
                else:
                    print(f"   ✈️  Not a numpy array: {type(away_players)}")
            else:
                print(f"   ✈️  Away players is None")
        else:
            print(f"   ✈️  No 'awayTeam_players' column found")
    
    print(f"\n📊 Debug Results:")
    print(f"   Total players found: {len(all_player_stats)}")
    
    return all_player_stats

# Test on first file
test_file = 'data/raw/Boxscores/2025_0042400176_NBA_Traditional_boxscores.parquet'
if Path(test_file).exists():
    debug_results = debug_parse_single_file(test_file)
else:
    print(f"❌ Test file not found: {test_file}")

In [ ]:
# Let's test the exact parsing function to see where it fails
print("🔧 TESTING EXACT PARSING FUNCTION")
print("=" * 40)

# Test the actual parse_boxscore_data function on the same file
test_file = 'data/raw/Boxscores/2025_0042400176_NBA_Traditional_boxscores.parquet'

if Path(test_file).exists():
    print("📊 Testing actual parse_boxscore_data function...")
    
    # Call the actual function
    result = parse_boxscore_data(test_file)
    
    print(f"\n📋 Results from parse_boxscore_data:")
    for key, df in result.items():
        if key != 'raw_data':
            print(f"   • {key}: {df.shape}")
            
    # Check player_stats specifically
    if 'player_stats' in result:
        player_df = result['player_stats']
        print(f"\n👤 Player stats details:")
        print(f"   Shape: {player_df.shape}")
        print(f"   Columns: {list(player_df.columns) if len(player_df.columns) > 0 else 'No columns'}")
        
        if len(player_df) > 0:
            print(f"   Sample data:")
            display(player_df.head(3))
        else:
            print(f"   ❌ Player DataFrame is empty!")

else:
    print(f"❌ Test file not found: {test_file}")

In [ ]:
# Create a fixed version of the parsing function
def parse_boxscore_data_fixed(parquet_file):
    """
    FIXED version: Parse complex boxscore parquet data into readable DataFrames.
    """
    
    # Read the parquet file
    df = pd.read_parquet(parquet_file)
    
    result = {}
    all_game_info = []
    all_team_stats = []
    all_player_stats = []
    
    print(f"📊 Processing {len(df)} games from {Path(parquet_file).name}")
    
    for idx, row in df.iterrows():
        try:
            game_id = row.get('gameId')
            home_team_id = row.get('homeTeam_teamId')
            away_team_id = row.get('awayTeam_teamId')
            
            # Extract basic game information
            game_info = {
                'game_id': game_id,
                'home_team_id': home_team_id,
                'away_team_id': away_team_id,
                'file_source': Path(parquet_file).name
            }
            all_game_info.append(game_info)
            
            # Extract team statistics (simplified)
            home_stats = {'game_id': game_id, 'team_type': 'home', 'team_id': home_team_id}
            away_stats = {'game_id': game_id, 'team_type': 'away', 'team_id': away_team_id}
            
            # Add team statistics
            for key, value in row.items():
                if key.startswith('homeTeam_statistics_'):
                    stat_name = key.replace('homeTeam_statistics_', '')
                    home_stats[stat_name] = value
                elif key.startswith('awayTeam_statistics_'):
                    stat_name = key.replace('awayTeam_statistics_', '')
                    away_stats[stat_name] = value
            
            all_team_stats.extend([home_stats, away_stats])
            
            # Extract player statistics - FIXED VERSION
            print(f"🔍 Row {idx}: Processing players...")
            
            # Home team players
            if 'homeTeam_players' in row and row['homeTeam_players'] is not None:
                home_players_array = row['homeTeam_players']
                print(f"   🏠 Found home players array: {type(home_players_array)}, length: {len(home_players_array) if hasattr(home_players_array, '__len__') else 'N/A'}")
                
                # Convert numpy array to list if needed
                if isinstance(home_players_array, np.ndarray):
                    home_players_list = home_players_array.tolist()
                else:
                    home_players_list = home_players_array if home_players_array else []
                
                print(f"   🏠 Processing {len(home_players_list)} home players...")
                
                # Process each player
                for i, player in enumerate(home_players_list):
                    if isinstance(player, dict):
                        player_stats = {
                            'game_id': game_id,
                            'team_type': 'home',
                            'team_id': home_team_id,
                            'player_id': player.get('personId'),
                            'player_name': player.get('name'),
                            'first_name': player.get('firstName'),
                            'family_name': player.get('familyName'),
                            'jersey_num': player.get('jerseyNum'),
                            'position': player.get('position'),
                            'starter': player.get('starter'),
                            'played': player.get('played'),
                            'status': player.get('status'),
                            'order': player.get('order')
                        }
                        
                        # Add player statistics
                        if 'statistics' in player and isinstance(player['statistics'], dict):
                            for stat_name, stat_value in player['statistics'].items():
                                player_stats[stat_name] = stat_value
                        
                        all_player_stats.append(player_stats)
                        print(f"      Added player {i+1}: {player.get('name', 'Unknown')}")
                    else:
                        print(f"      ⚠️ Player {i+1} is not a dict: {type(player)}")
            else:
                print(f"   🏠 No home players found")
            
            # Away team players
            if 'awayTeam_players' in row and row['awayTeam_players'] is not None:
                away_players_array = row['awayTeam_players']
                print(f"   ✈️  Found away players array: {type(away_players_array)}, length: {len(away_players_array) if hasattr(away_players_array, '__len__') else 'N/A'}")
                
                # Convert numpy array to list if needed
                if isinstance(away_players_array, np.ndarray):
                    away_players_list = away_players_array.tolist()
                else:
                    away_players_list = away_players_array if away_players_array else []
                
                print(f"   ✈️  Processing {len(away_players_list)} away players...")
                
                # Process each player
                for i, player in enumerate(away_players_list):
                    if isinstance(player, dict):
                        player_stats = {
                            'game_id': game_id,
                            'team_type': 'away',
                            'team_id': away_team_id,
                            'player_id': player.get('personId'),
                            'player_name': player.get('name'),
                            'first_name': player.get('firstName'),
                            'family_name': player.get('familyName'),
                            'jersey_num': player.get('jerseyNum'),
                            'position': player.get('position'),
                            'starter': player.get('starter'),
                            'played': player.get('played'),
                            'status': player.get('status'),
                            'order': player.get('order')
                        }
                        
                        # Add player statistics
                        if 'statistics' in player and isinstance(player['statistics'], dict):
                            for stat_name, stat_value in player['statistics'].items():
                                player_stats[stat_name] = stat_value
                        
                        all_player_stats.append(player_stats)
                        print(f"      Added player {i+1}: {player.get('name', 'Unknown')}")
                    else:
                        print(f"      ⚠️ Player {i+1} is not a dict: {type(player)}")
            else:
                print(f"   ✈️  No away players found")
                    
        except Exception as e:
            print(f"⚠️ Error processing row {idx}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    # Create DataFrames
    result['game_info'] = pd.DataFrame(all_game_info)
    result['team_stats'] = pd.DataFrame(all_team_stats)
    result['player_stats'] = pd.DataFrame(all_player_stats)
    result['raw_data'] = df
    
    # Print summary
    print(f"\n✅ Parsing complete!")
    print(f"   📋 Games processed: {len(result['game_info'])}")
    print(f"   🏀 Team stat records: {len(result['team_stats'])}")
    print(f"   👤 Player stat records: {len(result['player_stats'])}")
    
    return result

# Test the fixed function
print("🔧 TESTING FIXED PARSING FUNCTION")
print("=" * 40)

test_file = 'data/raw/Boxscores/2025_0042400176_NBA_Traditional_boxscores.parquet'
if Path(test_file).exists():
    fixed_result = parse_boxscore_data_fixed(test_file)
    
    print(f"\n📊 Fixed function results:")
    for key, df in fixed_result.items():
        if key != 'raw_data':
            print(f"   • {key}: {df.shape}")
    
    if len(fixed_result['player_stats']) > 0:
        print(f"\n👤 Sample player data:")
        sample_cols = ['player_name', 'team_type', 'position', 'points', 'assists']
        available_cols = [col for col in sample_cols if col in fixed_result['player_stats'].columns]
        if available_cols:
            display(fixed_result['player_stats'][available_cols].head(5))
else:
    print(f"❌ Test file not found")

In [ ]:
# Test the updated parsing function
print("🔧 TESTING UPDATED PARSING FUNCTION")
print("=" * 40)

test_file = 'data/raw/Boxscores/2025_0042400176_NBA_Traditional_boxscores.parquet'
if Path(test_file).exists():
    # Test the updated function
    updated_result = parse_boxscore_data(test_file)
    
    print(f"\n📊 Updated function results:")
    for key, df in updated_result.items():
        if key != 'raw_data':
            print(f"   • {key}: {df.shape}")
    
    # Check if player stats were extracted
    if len(updated_result['player_stats']) > 0:
        print(f"\n✅ SUCCESS! Player stats extracted!")
        player_df = updated_result['player_stats']
        
        print(f"👤 Player stats sample:")
        sample_cols = ['player_name', 'team_type', 'position', 'points', 'assists', 'reboundsTotal', 'steals', 'blocks', 'turnovers',
            'fieldGoalsMade', 'fieldGoalsAttempted', 'fieldGoalsPercentage',
            'threePointersMade', 'threePointersAttempted', 'threePointersPercentage']
        available_cols = [col for col in sample_cols if col in player_df.columns]
        if available_cols:
            display(player_df[available_cols].head(5))
            
        print(f"\n📊 Statistical columns available:")
        stat_cols = [col for col in player_df.columns if col in [
            'points', 'assists', 'reboundsTotal', 'steals', 'blocks', 'turnovers',
            'fieldGoalsMade', 'fieldGoalsAttempted', 'fieldGoalsPercentage',
            'threePointersMade', 'threePointersAttempted', 'threePointersPercentage'
        ]]
        for col in stat_cols:
            print(f"   • {col}")
    else:
        print(f"\n❌ Still no player stats extracted!")
        
else:
    print(f"❌ Test file not found")